In [18]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

# --------------------------------------------------
# 1. Define metadata files
# --------------------------------------------------
model_files = {
    "GPT-2": "data_output/bk21_with_gpt2_metadata.pt",
    "Gemma-270m": "data_output/bk21_with_gemma270m_metadata.pt",
    "Gemma-12b": "data_output/bk21_with_gemma12b_metadata.pt",
    "GPT-2 (NLTK)": "data_output/bk21_with_gpt2_nltk_metadata.pt",
    "Gemma-270m (NLTK)": "data_output/bk21_with_gemma270m_nltk_metadata.pt",
    "Gemma-12b (NLTK)": "data_output/bk21_with_gemma12b_nltk_metadata.pt",
}

In [19]:
# --------------------------------------------------
# 2. Load and flatten metadata
# --------------------------------------------------
token_rows = []
item_rows = []
summary_rows = []

for model_name, path in model_files.items():
    if not os.path.exists(path):
        print(f"Missing file: {path}")
        continue
    
    meta = torch.load(path, weights_only=False)
    
    vocab_type = "NLTK" if "NLTK" in model_name else "Full"
    base_model = model_name.replace(" (NLTK)", "")
    
    print(f"\n=== {model_name} ===")
    print(type(meta))
    print(f"Number of items: {len(meta)}")
    if len(meta) > 0:
        print("Keys:", meta[0].keys())
    
    for entry in meta:
        # item-level table
        item_rows.append({
            "model": model_name,
            "model_base": base_model,
            "vocab_type": vocab_type,
            "index": entry["index"],
            "uni_surprisal": entry["uni_surprisal"],
            "bi_surprisal": entry["bi_surprisal"],
            "n_target_tokens": len(entry["token_data"]),
            "model_id": entry["model_id"],
        })
        
        # token-level table
        for j, tok in enumerate(entry["token_data"]):
            token_rows.append({
                "model": model_name,
                "model_base": base_model,
                "vocab_type": vocab_type,
                "index": entry["index"],
                "model_id": entry["model_id"],
                "uni_surprisal_total": entry["uni_surprisal"],
                "bi_surprisal_total": entry["bi_surprisal"],
                "token_position_in_target": j,
                "token_id": tok["token_id"],
                "token_str": tok["token_str"],
                "token_surprisal": tok["surprisal"],
                "token_entropy": tok["entropy"],
            })
    
    summary_rows.append({
        "model": model_name,
        "n_items": len(meta),
        "n_unique_target_tokens": len({
            tok["token_id"] for entry in meta for tok in entry["token_data"]
        }),
        "mean_uni": sum(entry["uni_surprisal"] for entry in meta) / len(meta) if meta else float("nan"),
        "mean_bi": sum(entry["bi_surprisal"] for entry in meta) / len(meta) if meta else float("nan"),
    })

token_df = pd.DataFrame(token_rows)
item_df = pd.DataFrame(item_rows)
summary_df = pd.DataFrame(summary_rows)

print("\nToken-level dataframe shape:", token_df.shape)
print("Item-level dataframe shape:", item_df.shape)

display(summary_df)
display(token_df.head())
display(item_df.head())


=== GPT-2 ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

=== Gemma-270m ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

=== Gemma-12b ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

=== GPT-2 (NLTK) ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

=== Gemma-270m (NLTK) ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

=== Gemma-12b (NLTK) ===
<class 'list'>
Number of items: 648
Keys: dict_keys(['index', 'uni_surprisal', 'bi_surprisal', 'token_data', 'model_id'])

Token-level dataframe shape: (3990, 12)
Item-level dataframe shape: (3888, 8)


,model,n_items,n_unique_target_tokens,mean_uni,mean_bi
0,GPT-2,648,229,6.702654,4.835779
1,Gemma-270m,648,218,5.859559,4.126773
2,Gemma-12b,648,218,4.551277,2.791580
3,GPT-2 (NLTK),648,229,6.496771,5.862839
4,Gemma-270m (NLTK),648,218,5.646477,4.166658
5,Gemma-12b (NLTK),648,218,4.387051,2.821666


,model,model_base,vocab_type,index,model_id,uni_surprisal_total,bi_surprisal_total,token_position_in_target,token_id,token_str,token_surprisal,token_entropy
0,GPT-2,GPT-2,Full,0,gpt2,0.621024,0.174058,0,1633,air,0.621024,3.661430
1,GPT-2,GPT-2,Full,1,gpt2,2.350095,0.722724,0,1633,air,2.350095,8.653419
2,GPT-2,GPT-2,Full,2,gpt2,8.274212,4.558812,0,1633,air,8.274212,11.728687
3,GPT-2,GPT-2,Full,3,gpt2,3.252842,3.287098,0,5428,army,3.252842,8.132433
4,GPT-2,GPT-2,Full,4,gpt2,6.287171,4.464184,0,5428,army,6.287171,10.810644


,model,model_base,vocab_type,index,uni_surprisal,bi_surprisal,n_target_tokens,model_id
0,GPT-2,GPT-2,Full,0,0.621024,0.174058,1,gpt2
1,GPT-2,GPT-2,Full,1,2.350095,0.722724,1,gpt2
2,GPT-2,GPT-2,Full,2,8.274212,4.558812,1,gpt2
3,GPT-2,GPT-2,Full,3,3.252842,3.287098,1,gpt2
4,GPT-2,GPT-2,Full,4,6.287171,4.464184,1,gpt2
